# Compile VLASS Photometry

Compile the VLASS photometry from the cirada cutouts

In [3]:
import os
import glob
import pandas as pd
import numpy as np
from astropy.time import Time

In [4]:
txts = glob.glob('*/*.txt')
hdr_names = ['day', 'filter_eff', 'spectral_type', 'ra', 'dra', 'dec', 'ddec', 'flux', 'flux_err', 'rms']
data = []
for txt in txts:
    df = pd.read_csv(txt, names=hdr_names, delimiter=', ', parse_dates=True, engine='python')

    ecle_name = os.path.dirname(txt)
    df['name'] = [ecle_name]*len(df)
    df['filename'] = txt
    filename = os.path.basename(txt)

    if "QL_" in filename:
        start = filename.index('QL_')+3
    else:
        start = filename.index('SE_')+3

    vlass_version = filename[start:start+3]
    df['VLASS_version'] = vlass_version

    if vlass_version == '1.1':
        # we need to multiply by 1.15 and increase the dF by a 
        # factor of 1.08
        # https://science.nrao.edu/vlass/data-access/vlass-epoch-1-quick-look-users-guide
        df['flux'] *= 1.15
        df['flux_err'] *= 1.08

    df['flux_unit'] = 'mJy'
    df['filter'] = 'S'
    df['filter_eff_units'] = 'GHz'
    df['reducer'] = 'VLASS '+vlass_version
    df['source'] = 'VLASS '+vlass_version

    # convert date to the yday astropy Time format
    df['day'] = [date.replace('/','-',2).replace('/','T') for date in df['day']]
    df['date_mjd'] = [Time(date, format='isot').mjd for date in df['day']]

    data.append(df)

data = pd.concat(data).reset_index(drop=True)
datai = data[data.spectral_type == 'I']

In [5]:
datai['upperlimit'] = datai.apply(lambda row: row.flux_err*3 >= row.flux if row.flux_err>0 and row.flux>0 else True , axis=1)

datai['flux'] = datai.apply(lambda row : 3*row.rms if row.upperlimit else row.flux, axis=1)
datai['flux_err'] = datai.apply(lambda row: 0 if row.upperlimit else row.flux_err, axis=1)

/tmp/ipykernel_174415/2817651559.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datai['upperlimit'] = datai.apply(lambda row: row.flux_err*3 >= row.flux if row.flux_err>0 and row.flux>0 else True , axis=1)
/tmp/ipykernel_174415/2817651559.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datai['flux'] = datai.apply(lambda row : 3*row.rms if row.upperlimit else row.flux, axis=1)
/tmp/ipykernel_174415/2817651559.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

In [6]:
sheet_keys = ['name','day','date_mjd','filter','filter_eff','filter_eff_units','flux','flux_err','flux_unit','upperlimit','reducer','source']
datai[sheet_keys].to_csv('vlass-for-spreadsheet.csv', index=False)

In [7]:
datai

,day,filter_eff,spectral_type,ra,dra,dec,ddec,flux,flux_err,rms,name,filename,VLASS_version,flux_unit,filter,filter_eff_units,reducer,source,date_mjd,upperlimit
0,2017-10-06T11:18:25.425000,3.0,I,2.126936,0.433074,0.245976,0.413266,0.335159,0.000000,0.111720,SDSS_J0807,SDSS_J0807/SDSS_J0807-08-VLASS-QL_1.1_J121.86+...,1.1,mJy,S,GHz,VLASS 1.1,VLASS 1.1,58032.471128,True
5,2017-10-06T11:18:25.425000,3.0,I,2.126936,0.433074,0.245976,0.413266,0.335159,0.000000,0.111720,SDSS_J0807,SDSS_J0807/SDSS_J0807-08-VLASS-QL_1.1_J121.86+...,1.1,mJy,S,GHz,VLASS 1.1,VLASS 1.1,58032.471128,True
10,2017-10-06T11:18:25.425000,3.0,I,2.126936,0.474931,0.245976,0.453209,0.334668,0.000000,0.111556,SDSS_J0807,SDSS_J0807/SDSS_J0807-08-VLASS-QL_1.1_J121.86+...,1.1,mJy,S,GHz,VLASS 1.1,VLASS 1.1,58032.471128,True
15,2020-08-18T17:12:15.750000,3.0,I,2.126907,0.308507,0.245986,0.385383,0.519151,0.170478,0.168421,SDSS_J0807,SDSS_J0807/SDSS_J0807-08-VLASS-QL_2.1_J121.86+...,2.1,mJy,S,GHz,VLASS 2.1,VLASS 2.1,59079.716849,False
20,2020-08-18T17:12:15.750000,3.0,I,2.126907,0.308507,0.245986,0.385383,0.519151,0.170478,0.168421,SDSS_J0807,SDSS_J0807/SDSS_J0807-08-VLASS-QL_2.1_J121.86+...,2.1,mJy,S,GHz,VLASS 2.1,VLASS 2.1,59079.716849,False
25,2020-08-18T17:12:15.750000,3.0,I,2.126907,0.304559,0.245986,0.380451,0.519151,0.168296,0.159502,SDSS_J0807,SDSS_J0807/SDSS_J0807-08-VLASS-QL_2.1_J121.86+...,2.1,mJy,S,GHz,VLASS 2.1,VLASS 2.1,59079.716849,False
30,2023-01-29T09:32:29.400001,3.0,I,2.126931,0.662932,0.245984,0.614544,0.398643,0.000000,0.132881,SDSS_J0807,SDSS_J0807/SDSS_J0807-08-VLASS-QL_3.1_J121.86+...,3.1,mJy,S,GHz,VLASS 3.1,VLASS 3.1,59973.397563,True
35,2023-01-29T09:32:29.400001,3.0,I,2.126931,0.662932,0.245984,0.614544,0.398643,0.000000,0.132881,SDSS_J0807,SDSS_J0807/SDSS_J0807-08-VLASS-QL_3.1_J121.86+...,3.1,mJy,S,GHz,VLASS 3.1,VLASS 3.1,59973.397563,True
40,2023-01-29T09:32:29.400001,3.0,I,2.126931,0.657852,0.245984,0.609835,0.361806,0.000000,0.120602,SDSS_J0807,SDSS_J0807/SDSS_J0807-08-VLASS-QL_3.1_J121.86+...,3.1,mJy,S,GHz,VLASS 3.1,VLASS 3.1,59973.397563,True
45,2020-09-18T00:05:56.400001,3.0,I,-2.608925,0.658571,0.517744,0.640099,0.445452,0.000000,0.148484,SDSS_J1402,SDSS_J1402/SDSS_J1402-11-VLASS-QL_2.1_J210.52+...,2.1,mJy,S,GHz,VLASS 2.1,VLASS 2.1,59110.004125,True
